# ETL do conjunto de dados do Banco de Preços em Saúde

Análise estrutural dos arquivos anuais do BPS (2020 a 2026), realizada antes da concatenação.

In [ ]:
import glob
from pathlib import Path

import pandas as pd

identifier_columns = [
    'codigo_br',
    'anvisa',
    'cnpj_instituicao',
    'cnpj_fornecedor',
    'cnpj_fabricante',
]

# Funciona quando o Jupyter é iniciado na raiz ou na pasta notebooks.
project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent

data_dir = project_dir / 'data'
path_files = sorted(glob.glob(str(data_dir / '*.csv')))

if not path_files:
    raise FileNotFoundError(f'Nenhum arquivo CSV encontrado em: {data_dir}')

print(f'{len(path_files)} arquivos encontrados:')
for file_path in path_files:
    print(f'- {Path(file_path).name}')

## 1. Análise estrutural antes da concatenação

Cada arquivo é carregado separadamente. O resumo registra dimensões, nulos e duplicatas, enquanto as colunas e os tipos inferidos são preservados para comparação entre os anos.

In [ ]:
dataframes = {}
columns_by_file = {}
dtypes_by_file = {}
structural_records = []

for file_path in path_files:
    path = Path(file_path)
    df = pd.read_csv(
        path,
        sep=';',
        encoding='utf-8',
        low_memory=False,
        dtype={column: 'string' for column in identifier_columns},
    )

    null_count = int(df.isna().sum().sum())
    total_cells = df.shape[0] * df.shape[1]

    structural_records.append({
        'arquivo': path.name,
        'ano': path.stem,
        'linhas': df.shape[0],
        'colunas': df.shape[1],
        'valores_nulos': null_count,
        'percentual_nulos': null_count / total_cells if total_cells else 0,
        'colunas_com_nulos': int(df.isna().any().sum()),
        'linhas_duplicadas': int(df.duplicated().sum())
    })

    columns_by_file[path.name] = df.columns.tolist()
    dtypes_by_file[path.name] = df.dtypes.astype(str).to_dict()
    df['arquivo_origem'] = path.name
    dataframes[path.name] = df

structural_summary = pd.DataFrame(structural_records)

structural_summary['percentual_nulos'] = (
    structural_summary['percentual_nulos'] * 100
).round(2)

display(structural_summary)

In [ ]:
reference_file = Path(path_files[0]).name
reference_columns = columns_by_file[reference_file]
reference_dtypes = dtypes_by_file[reference_file]
schema_records = []

for file_name in dataframes:
    current_columns = columns_by_file[file_name]
    current_dtypes = dtypes_by_file[file_name]
    missing_columns = sorted(set(reference_columns) - set(current_columns))
    extra_columns = sorted(set(current_columns) - set(reference_columns))
    common_columns = set(reference_columns) & set(current_columns)
    divergent_types = sorted(
        column
        for column in common_columns
        if current_dtypes[column] != reference_dtypes[column]
    )

    schema_records.append({
        'arquivo': file_name,
        'mesmas_colunas': not missing_columns and not extra_columns,
        'mesma_ordem': current_columns == reference_columns,
        'colunas_ausentes': ', '.join(missing_columns) or '-',
        'colunas_extras': ', '.join(extra_columns) or '-',
        'tipos_divergentes': ', '.join(divergent_types) or '-',
    })

schema_comparison = pd.DataFrame(schema_records)
type_matrix = pd.DataFrame(dtypes_by_file).sort_index()

print(f'Arquivo de referência para a comparação: {reference_file}')
display(schema_comparison)
display(type_matrix)

if schema_comparison['mesmas_colunas'].all():
    print('Estruturas compatíveis: todos os arquivos possuem o mesmo conjunto de colunas.')
else:
    print('Atenção: existem colunas ausentes ou adicionais. Revise antes de concatenar.')

## 2. Concatenação das bases

A concatenação só é executada quando todos os arquivos apresentam o mesmo conjunto de colunas. Diferenças de ordem não causam perda, pois o pandas alinha os dados pelo nome das colunas.

In [ ]:
if not schema_comparison['mesmas_colunas'].all():
    raise ValueError('Concatenação interrompida: foram encontradas estruturas incompatíveis.')

bps_raw = pd.concat(dataframes.values(), ignore_index=True, sort=False)

print(f'{len(dataframes)} arquivos concatenados.')
print(f'{len(bps_raw):,} registros e {bps_raw.shape[1]} colunas na base bruta consolidada.')
display(bps_raw.head())

## 3. Diagnóstico de qualidade

O dicionário oficial define `compra` e `insercao` como datas; `codigo_br`, `anvisa` e CNPJs como identificadores; `capacidade`, `qtd_itens_comprados`, `preco_unitario` e `preco_total` como medidas numéricas. A base também possui `esfera` e `unidade_fornecimento_capacidade`, que não estão descritas de forma inequívoca no documento de metadados.

A base bruta é preservada em `bps_raw`. As transformações serão aplicadas somente a uma cópia chamada `bps_clean`.

In [ ]:
date_columns = ['compra', 'insercao']
numeric_columns = [
    'capacidade',
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]
cnpj_columns = [
    'cnpj_instituicao',
    'cnpj_fornecedor',
    'cnpj_fabricante',
]

null_report_raw = (
    pd.DataFrame({
        'valores_nulos': bps_raw.isna().sum(),
        'percentual_nulos': bps_raw.isna().mean().mul(100).round(2),
    })
    .query('valores_nulos > 0')
    .sort_values('percentual_nulos', ascending=False)
)

diagnostic_records = []

for column in date_columns:
    parsed = pd.to_datetime(bps_raw[column], format='%d/%m/%Y', errors='coerce')
    diagnostic_records.append({
        'verificacao': f'{column}: datas não convertíveis',
        'registros': int((bps_raw[column].notna() & parsed.isna()).sum()),
    })

for column in numeric_columns:
    converted = pd.to_numeric(bps_raw[column], errors='coerce')
    diagnostic_records.extend([
        {
            'verificacao': f'{column}: valores não numéricos',
            'registros': int((bps_raw[column].notna() & converted.isna()).sum()),
        },
        {
            'verificacao': f'{column}: valores negativos',
            'registros': int(converted.lt(0).sum()),
        },
    ])

for column in cnpj_columns:
    digits = bps_raw[column].str.replace(r'\D', '', regex=True)
    diagnostic_records.append({
        'verificacao': f'{column}: tamanho diferente de 14 dígitos',
        'registros': int((digits.notna() & digits.str.len().ne(14)).sum()),
    })

raw_duplicates = int(bps_raw.duplicated().sum())
diagnostic_records.append({
    'verificacao': 'linhas exatamente duplicadas',
    'registros': raw_duplicates,
})

diagnostic_report = pd.DataFrame(diagnostic_records)
display(null_report_raw)
display(diagnostic_report)

## 4. Limpeza auditável

As regras abaixo preservam os dados originais, não imputam medidas ausentes e não descartam anomalias que exigem investigação. Duplicatas exatas são removidas da cópia limpa; datas suspeitas e divergências de valor recebem indicadores de qualidade.

In [ ]:
bps_clean = bps_raw.copy()
treatment_records = []

# Normaliza espaços em campos textuais sem alterar maiúsculas, acentos ou conteúdo.
text_columns = bps_clean.select_dtypes(include=['object', 'string']).columns
for column in text_columns:
    original = bps_clean[column].astype('string')
    normalized = original.str.replace(r'\s+', ' ', regex=True).str.strip()
    changed = int((original.notna() & original.ne(normalized)).sum())
    bps_clean[column] = normalized
    if changed:
        treatment_records.append({
            'tratamento': 'normalização de espaços',
            'coluna': column,
            'registros_afetados': changed,
        })

# Mantém CNPJs como texto de 14 dígitos, preservando zeros à esquerda.
for column in cnpj_columns:
    normalized = bps_clean[column].str.replace(r'\D', '', regex=True)
    invalid_mask = normalized.notna() & normalized.str.len().ne(14)
    changed = int(bps_clean[column].notna().sum())
    bps_clean[column] = normalized.mask(invalid_mask)
    treatment_records.append({
        'tratamento': 'CNPJ convertido para 14 dígitos',
        'coluna': column,
        'registros_afetados': changed,
    })

# Converte datas e medidas conforme o dicionário oficial.
for column in date_columns:
    bps_clean[column] = pd.to_datetime(
        bps_clean[column],
        format='%d/%m/%Y',
        errors='coerce',
    )

for column in numeric_columns:
    bps_clean[column] = pd.to_numeric(bps_clean[column], errors='coerce')

bps_clean['ano_compra'] = pd.to_numeric(
    bps_clean['ano_compra'], errors='coerce'
).astype('Int64')

# Preenche apenas rótulos categóricos úteis ao dashboard; medidas ficam nulas.
categorical_fill = {
    'nome_instituicao': 'INSTITUIÇÃO NÃO INFORMADA',
    'generico': 'NÃO INFORMADO',
    'unidade_fornecimento': 'NÃO INFORMADO',
    'unidade_fornecimento_capacidade': 'NÃO INFORMADO',
}
for column, replacement in categorical_fill.items():
    affected = int(bps_clean[column].isna().sum())
    bps_clean[column] = bps_clean[column].fillna(replacement)
    treatment_records.append({
        'tratamento': f'nulo substituído por {replacement}',
        'coluna': column,
        'registros_afetados': affected,
    })

# Remove somente duplicatas exatas, mantendo a primeira ocorrência.
duplicates_after_normalization = int(bps_clean.duplicated().sum())
bps_clean = bps_clean.drop_duplicates().reset_index(drop=True)
treatment_records.append({
    'tratamento': 'remoção de duplicatas exatas',
    'coluna': 'todas',
    'registros_afetados': duplicates_after_normalization,
})

# Cria indicadores para anomalias que devem ser investigadas, não apagadas.
bps_clean['flag_insercao_anterior_compra'] = (
    bps_clean['insercao'].notna()
    & bps_clean['compra'].notna()
    & bps_clean['insercao'].lt(bps_clean['compra'])
)

calculated_total = bps_clean['qtd_itens_comprados'] * bps_clean['preco_unitario']
price_tolerance = 0.01 + bps_clean['preco_total'].abs() * 1e-6
bps_clean['flag_preco_total_divergente'] = (
    bps_clean[['qtd_itens_comprados', 'preco_unitario', 'preco_total']]
    .notna()
    .all(axis=1)
    & bps_clean['preco_total'].sub(calculated_total).abs().gt(price_tolerance)
)

treatment_log = pd.DataFrame(treatment_records)
display(treatment_log)

## 5. Validação da base limpa

In [ ]:
validation_report = pd.DataFrame([
    {
        'verificacao': 'registros na base bruta',
        'resultado': len(bps_raw),
    },
    {
        'verificacao': 'duplicatas removidas',
        'resultado': duplicates_after_normalization,
    },
    {
        'verificacao': 'registros na base limpa',
        'resultado': len(bps_clean),
    },
    {
        'verificacao': 'datas da compra inválidas',
        'resultado': int(bps_clean['compra'].isna().sum()),
    },
    {
        'verificacao': 'ano divergente da data da compra',
        'resultado': int((
            bps_clean['compra'].notna()
            & bps_clean['ano_compra'].notna()
            & bps_clean['compra'].dt.year.ne(bps_clean['ano_compra'])
        ).sum()),
    },
    {
        'verificacao': 'inserções anteriores à compra',
        'resultado': int(bps_clean['flag_insercao_anterior_compra'].sum()),
    },
    {
        'verificacao': 'preços totais divergentes da fórmula',
        'resultado': int(bps_clean['flag_preco_total_divergente'].sum()),
    },
    {
        'verificacao': 'quantidades ou preços negativos',
        'resultado': int(
            bps_clean[['qtd_itens_comprados', 'preco_unitario', 'preco_total']]
            .lt(0)
            .any(axis=1)
            .sum()
        ),
    },
])

assert len(bps_clean) == len(bps_raw) - duplicates_after_normalization
assert not bps_clean.duplicated().any()
assert validation_report.loc[
    validation_report['verificacao'] == 'ano divergente da data da compra',
    'resultado',
].iat[0] == 0
assert not bps_clean['flag_preco_total_divergente'].any()

# Atalho para as próximas etapas de análise e visualização.
bps = bps_clean

display(validation_report)
display(bps_clean.head())

# Análise exploratória dos dados

## 6. Perguntas de negócio

1. Como o valor e a quantidade registrados evoluíram entre 2020 e 2026?
2. Quais estados, municípios e instituições concentram o maior volume financeiro?
3. Quais produtos apresentam maior valor total e maior quantidade adquirida?
4. Quais fornecedores e fabricantes possuem maior participação nos registros?
5. Quais modalidades e tipos de compra são mais utilizados?
6. Quais produtos comparáveis apresentam maior variação de preço unitário?
7. Quais registros merecem investigação adicional sem pressupor irregularidade?

## 7. Visão geral e cobertura dos dados

A cobertura temporal é conferida antes das comparações. O ano de 2026 é parcial e não deve ser comparado diretamente a anos completos sem essa ressalva.

In [ ]:
measure_columns = [
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]

numeric_summary = (
    bps[measure_columns]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.99])
    .round(2)
    .T
)

temporal_coverage = (
    bps.groupby('ano_compra', observed=True)
    .agg(
        primeira_compra=('compra', 'min'),
        ultima_compra=('compra', 'max'),
        registros=('compra', 'size'),
    )
    .reset_index()
)
temporal_coverage['ano_parcial'] = (
    temporal_coverage['ultima_compra'].dt.month.lt(12)
    | temporal_coverage['ultima_compra'].dt.day.lt(28)
)

cardinality_report = pd.DataFrame({
    'dimensao': [
        'UFs',
        'municípios',
        'instituições',
        'produtos CATMAT',
        'fornecedores',
        'fabricantes',
        'modalidades de compra',
    ],
    'quantidade_distinta': [
        bps['uf'].nunique(),
        bps[['uf', 'municipio_instituicao']].drop_duplicates().shape[0],
        bps['cnpj_instituicao'].nunique(),
        bps['codigo_br'].nunique(),
        bps['cnpj_fornecedor'].nunique(),
        bps['cnpj_fabricante'].nunique(),
        bps['modalidade_compra'].nunique(),
    ],
})

display(numeric_summary)
display(temporal_coverage)
display(cardinality_report)

## 8. KPIs obrigatórios

O preço unitário médio ponderado é calculado como valor total dividido pela quantidade total. Ele só deve ser interpretado dentro de filtros que mantenham produtos, apresentações e unidades comparáveis.

In [ ]:
total_value = bps['preco_total'].sum()
total_quantity = bps['qtd_itens_comprados'].sum()
weighted_unit_price = total_value / total_quantity if total_quantity else pd.NA

kpis = pd.DataFrame([
    {
        'kpi': 'Valor total registrado',
        'valor': total_value,
        'unidade': 'R$',
        'definicao': 'Soma de preco_total',
    },
    {
        'kpi': 'Quantidade total de itens comprados',
        'valor': total_quantity,
        'unidade': 'itens',
        'definicao': 'Soma de qtd_itens_comprados',
    },
    {
        'kpi': 'Número de registros de compra',
        'valor': len(bps),
        'unidade': 'registros',
        'definicao': 'Contagem de linhas da base limpa',
    },
    {
        'kpi': 'Instituições compradoras',
        'valor': bps['cnpj_instituicao'].nunique(),
        'unidade': 'instituições distintas',
        'definicao': 'Contagem distinta de CNPJ da instituição',
    },
    {
        'kpi': 'Fornecedores',
        'valor': bps['cnpj_fornecedor'].nunique(),
        'unidade': 'fornecedores distintos',
        'definicao': 'Contagem distinta de CNPJ do fornecedor',
    },
    {
        'kpi': 'Preço unitário médio ponderado',
        'valor': weighted_unit_price,
        'unidade': 'R$/item',
        'definicao': 'Valor total dividido pela quantidade total',
    },
])

recomputed_total = (bps['qtd_itens_comprados'] * bps['preco_unitario']).sum()
reconciliation_tolerance = 0.01 + abs(total_value) * 1e-9
assert abs(total_value - recomputed_total) <= reconciliation_tolerance
assert int(kpis.loc[kpis['kpi'] == 'Número de registros de compra', 'valor'].iat[0]) == len(bps)

display(kpis)

## 9. Evolução anual

In [ ]:
annual_summary = (
    bps.groupby('ano_compra', observed=True)
    .agg(
        valor_total=('preco_total', 'sum'),
        quantidade_total=('qtd_itens_comprados', 'sum'),
        registros=('preco_total', 'size'),
        instituicoes=('cnpj_instituicao', 'nunique'),
        fornecedores=('cnpj_fornecedor', 'nunique'),
    )
    .reset_index()
)
annual_summary['preco_unitario_medio_ponderado'] = (
    annual_summary['valor_total'] / annual_summary['quantidade_total']
)
annual_summary = annual_summary.merge(
    temporal_coverage[['ano_compra', 'ultima_compra', 'ano_parcial']],
    on='ano_compra',
    how='left',
)

assert annual_summary['valor_total'].sum() == total_value
assert annual_summary['quantidade_total'].sum() == total_quantity
assert annual_summary['registros'].sum() == len(bps)

display(annual_summary)

## 10. Distribuição geográfica e instituições

Os rankings utilizam `preco_total` como volume financeiro registrado. Instituições são identificadas pelo CNPJ para evitar dupla contagem por variações no nome.

In [ ]:
def summarize_by_value(group_columns, top_n=10):
    return (
        bps.groupby(group_columns, dropna=False, observed=True)
        .agg(
            valor_total=('preco_total', 'sum'),
            quantidade_total=('qtd_itens_comprados', 'sum'),
            registros=('preco_total', 'size'),
        )
        .reset_index()
        .nlargest(top_n, 'valor_total')
    )

top_states = summarize_by_value(['uf'], top_n=27)
top_municipalities = summarize_by_value(
    ['uf', 'municipio_instituicao'], top_n=10
)
top_institutions = summarize_by_value(
    ['cnpj_instituicao', 'nome_instituicao'], top_n=10
)

display(top_states)
display(top_municipalities)
display(top_institutions)

## 11. Produtos, fornecedores, fabricantes e modalidades

In [ ]:
product_dimensions = [
    'codigo_br',
    'descricao_catmat',
    'unidade_fornecimento_capacidade',
]

product_summary = (
    bps.groupby(product_dimensions, dropna=False, observed=True)
    .agg(
        valor_total=('preco_total', 'sum'),
        quantidade_total=('qtd_itens_comprados', 'sum'),
        registros=('preco_total', 'size'),
        fornecedores=('cnpj_fornecedor', 'nunique'),
    )
    .reset_index()
)
top_products_by_value = product_summary.nlargest(10, 'valor_total')
top_products_by_quantity = product_summary.nlargest(10, 'quantidade_total')

top_suppliers = summarize_by_value(
    ['cnpj_fornecedor', 'fornecedor'], top_n=10
)
top_manufacturers = summarize_by_value(
    ['cnpj_fabricante', 'fabricante'], top_n=10
)
purchase_modes = summarize_by_value(['modalidade_compra'], top_n=20)
purchase_types = summarize_by_value(['tipo_compra'], top_n=20)

display(top_products_by_value)
display(top_products_by_quantity)
display(top_suppliers)
display(top_manufacturers)
display(purchase_modes)
display(purchase_types)

## 12. Variação de preços entre produtos comparáveis

A comparação agrupa o mesmo código CATMAT e a mesma unidade/capacidade. São considerados apenas grupos com pelo menos 10 registros, dois fornecedores e preços positivos. A classificação utiliza a razão entre os percentis 90 e 10, que é menos sensível a valores extremos do que máximo/mínimo. Uma razão elevada indica somente uma oportunidade de investigação; não comprova sobrepreço, economia ou irregularidade.

In [ ]:
price_variation = (
    bps.groupby(product_dimensions, dropna=False, observed=True)
    .agg(
        registros=('preco_unitario', 'size'),
        fornecedores=('cnpj_fornecedor', 'nunique'),
        quantidade_total=('qtd_itens_comprados', 'sum'),
        valor_total=('preco_total', 'sum'),
        preco_minimo=('preco_unitario', 'min'),
        preco_p10=('preco_unitario', lambda values: values.quantile(0.10)),
        preco_mediano=('preco_unitario', 'median'),
        preco_p90=('preco_unitario', lambda values: values.quantile(0.90)),
        preco_maximo=('preco_unitario', 'max'),
    )
    .reset_index()
)
price_variation['preco_medio_ponderado'] = (
    price_variation['valor_total'] / price_variation['quantidade_total']
)
price_variation['razao_max_min'] = (
    price_variation['preco_maximo'] / price_variation['preco_minimo']
)
price_variation['razao_p90_p10'] = (
    price_variation['preco_p90'] / price_variation['preco_p10']
)

price_investigation_candidates = (
    price_variation.loc[
        price_variation['registros'].ge(10)
        & price_variation['fornecedores'].ge(2)
        & price_variation['preco_minimo'].gt(0)
    ]
    .nlargest(20, 'razao_p90_p10')
)

assert price_investigation_candidates['registros'].ge(10).all()
assert price_investigation_candidates['fornecedores'].ge(2).all()
assert price_investigation_candidates['preco_minimo'].gt(0).all()
assert price_investigation_candidates['preco_p10'].gt(0).all()

display(price_investigation_candidates)

## 13. Limitações preliminares

- O BPS contém valores registrados pelas instituições e não deve ser interpretado automaticamente como o universo completo das compras em saúde.
- O ano de 2026 possui cobertura parcial até a última data observada na base.
- O preço unitário médio ponderado global mistura produtos e apresentações distintas; seu uso analítico exige filtros de comparabilidade.
- Valores extremos devem ser investigados juntamente com fabricante, fornecedor, modalidade, quantidade, localidade e período.
- Diferenças de preço não constituem, isoladamente, evidência de economia, sobrepreço ou irregularidade.
- Campos como `capacidade`, `unidade_medida`, `generico` e `anvisa` apresentam ausência relevante e limitam algumas comparações.

## 14. Exportação da base para o dashboard

A base tratada é exportada em CSV UTF-8, separado por vírgulas e com datas no padrão `YYYY-MM-DD`. O arquivo é gerado em `output/data/` e permanece fora do Git devido ao tamanho. O esquema versionado em `config/bigquery_schema.json` deve ser usado na carga para o BigQuery.

In [ ]:
import csv
import hashlib
import json

output_dir = project_dir / 'output' / 'data'
output_dir.mkdir(parents=True, exist_ok=True)
dashboard_file = output_dir / 'BPS_20_26_SamuelBucco.csv'
schema_file = project_dir / 'config' / 'bigquery_schema.json'

# O upload de CSV do Looker Studio não aceita quebras de linha internas.
dashboard_text_columns = bps.select_dtypes(include=['object', 'string']).columns
internal_line_breaks = {
    column: int(
        bps[column]
        .astype('string')
        .str.contains(r'[\r\n]', regex=True, na=False)
        .sum()
    )
    for column in dashboard_text_columns
}
internal_line_breaks = {
    column: count
    for column, count in internal_line_breaks.items()
    if count > 0
}
if internal_line_breaks:
    raise ValueError(
        f'Quebras de linha internas encontradas: {internal_line_breaks}'
    )

bps.to_csv(
    dashboard_file,
    index=False,
    encoding='utf-8',
    sep=',',
    date_format='%Y-%m-%d',
    na_rep='',
)

with schema_file.open(encoding='utf-8') as stream:
    bigquery_schema = json.load(stream)
schema_columns = [field['name'] for field in bigquery_schema]

with dashboard_file.open(encoding='utf-8', newline='') as stream:
    reader = csv.reader(stream)
    exported_header = next(reader)
    exported_rows = sum(1 for _ in reader)

sha256 = hashlib.sha256()
with dashboard_file.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        sha256.update(chunk)

file_size_mb = dashboard_file.stat().st_size / 1024**2
export_validation = pd.DataFrame([
    {'verificacao': 'arquivo gerado', 'resultado': str(dashboard_file)},
    {'verificacao': 'linhas exportadas', 'resultado': exported_rows},
    {'verificacao': 'colunas exportadas', 'resultado': len(exported_header)},
    {'verificacao': 'tamanho em MB', 'resultado': round(file_size_mb, 2)},
    {'verificacao': 'SHA-256', 'resultado': sha256.hexdigest()},
    {
        'verificacao': 'compatível com limite de 100 MB do upload direto',
        'resultado': file_size_mb <= 100,
    },
])

assert exported_rows == len(bps)
assert exported_header == bps.columns.tolist()
assert schema_columns == exported_header
assert len(exported_header) == bps.shape[1]

display(export_validation)

## 15. Exportação enxuta para Google Sheets

Para o projeto acadêmico, uma segunda fonte preserva somente os campos usados pelos KPIs, filtros e visuais do dashboard. A redução evita operar próximo ao limite de 10 milhões de células do Google Sheets sem substituir a base completa de auditoria.

In [ ]:
import numpy as np

sheets_source_columns = [
    'ano_compra',
    'compra',
    'uf',
    'municipio_instituicao',
    'cnpj_instituicao',
    'nome_instituicao',
    'esfera',
    'codigo_br',
    'descricao_catmat',
    'unidade_fornecimento_capacidade',
    'modalidade_compra',
    'tipo_compra',
    'cnpj_fornecedor',
    'fornecedor',
    'cnpj_fabricante',
    'fabricante',
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]

sheets_data = (
    bps.loc[:, sheets_source_columns]
    .rename(columns={
        'cnpj_instituicao': 'id_instituicao',
        'cnpj_fornecedor': 'id_fornecedor',
        'cnpj_fabricante': 'id_fabricante',
    })
    .copy()
)
sheets_data['id_instituicao'] = 'CNPJ ' + sheets_data['id_instituicao'].astype('string')
sheets_data['id_fornecedor'] = 'CNPJ ' + sheets_data['id_fornecedor'].astype('string')
sheets_data['id_fabricante'] = 'CNPJ ' + sheets_data['id_fabricante'].astype('string')
sheets_data['codigo_br'] = 'BR ' + sheets_data['codigo_br'].astype('string')
sheets_columns = [
    'ano_compra',
    'compra',
    'uf',
    'municipio_instituicao',
    'id_instituicao',
    'nome_instituicao',
    'esfera',
    'codigo_br',
    'descricao_catmat',
    'unidade_fornecimento_capacidade',
    'modalidade_compra',
    'tipo_compra',
    'id_fornecedor',
    'fornecedor',
    'id_fabricante',
    'fabricante',
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]
sheets_data = sheets_data.loc[:, sheets_columns]
sheets_file = output_dir / 'BPS_20_26_SamuelBucco_GoogleSheets.csv'
sheets_data.to_csv(
    sheets_file,
    index=False,
    encoding='utf-8',
    sep=',',
    date_format='%Y-%m-%d',
    na_rep='',
)

sheets_identifier_columns = [
    'id_instituicao',
    'codigo_br',
    'id_fornecedor',
    'id_fabricante',
]
sheets_loaded = pd.read_csv(
    sheets_file,
    dtype={column: 'string' for column in sheets_identifier_columns},
    parse_dates=['compra'],
)

sheets_limit_cells = 10_000_000
sheets_total_cells = (len(sheets_loaded) + 1) * len(sheets_loaded.columns)
sheets_occupancy = sheets_total_cells / sheets_limit_cells
sheets_weighted_price = (
    sheets_loaded['preco_total'].sum()
    / sheets_loaded['qtd_itens_comprados'].sum()
)
source_weighted_price = bps['preco_total'].sum() / bps['qtd_itens_comprados'].sum()

sheets_sha256 = hashlib.sha256()
with sheets_file.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        sheets_sha256.update(chunk)

assert sheets_loaded.columns.tolist() == sheets_columns
assert len(sheets_loaded) == len(bps)
assert sheets_total_cells < sheets_limit_cells
assert np.isclose(sheets_loaded['preco_total'].sum(), bps['preco_total'].sum())
assert sheets_loaded['qtd_itens_comprados'].sum() == bps['qtd_itens_comprados'].sum()
assert sheets_loaded['id_instituicao'].nunique() == bps['cnpj_instituicao'].nunique()
assert sheets_loaded['id_fornecedor'].nunique() == bps['cnpj_fornecedor'].nunique()
assert sheets_loaded['id_fabricante'].nunique() == bps['cnpj_fabricante'].nunique()
assert sheets_loaded['id_instituicao'].str.startswith('CNPJ ', na=False).all()
assert sheets_loaded['id_fornecedor'].str.startswith('CNPJ ', na=False).all()
assert sheets_loaded['id_fabricante'].str.startswith('CNPJ ', na=False).all()
assert sheets_loaded['codigo_br'].str.startswith('BR ', na=False).all()
assert np.isclose(sheets_weighted_price, source_weighted_price)

sheets_export_validation = pd.DataFrame([
    {'verificacao': 'arquivo gerado', 'resultado': str(sheets_file)},
    {'verificacao': 'linhas exportadas', 'resultado': len(sheets_loaded)},
    {'verificacao': 'colunas exportadas', 'resultado': len(sheets_loaded.columns)},
    {'verificacao': 'células com cabeçalho', 'resultado': sheets_total_cells},
    {'verificacao': 'ocupação do limite', 'resultado': f'{sheets_occupancy:.2%}'},
    {'verificacao': 'tamanho em MB', 'resultado': round(sheets_file.stat().st_size / 1024**2, 2)},
    {'verificacao': 'SHA-256', 'resultado': sheets_sha256.hexdigest()},
])

display(sheets_export_validation)